<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 1 - Tatiana

This document aims to identify duplicates in the project's image dataset.

## Identify empty, corrupted and duplicated images

In [3]:
import os
import hashlib
import pandas as pd
from PIL import Image

def get_image_details(directory):
    data = []

    # We use a dictionary to track hashes for duplicate detection
    # Key: MD5 hash, Value: List of file paths with that hash
    hashes = {}

    for root, _, files in os.walk(directory):
        for file in files:
            if not (file.lower().endswith(('.jpg', '.jpeg', '.png'))):
                continue

            file_path = os.path.join(root, file)
            file_size = os.path.getsize(file_path)

            is_empty = file_size == 0
            is_corrupted = False
            md5_hash = None

            if not is_empty:
                # 1. Calculate Hash for Duplicate Detection
                try:
                    with open(file_path, "rb") as f:
                        file_content = f.read()
                        md5_hash = hashlib.md5(file_content).hexdigest()
                except Exception:
                    md5_hash = "error_reading_hash"

                # 2. Check for Corruption using Pillow
                try:
                    with Image.open(file_path) as img:
                        img.verify()  # Verify integrity
                except (IOError, SyntaxError, Image.UnidentifiedImageError):
                    is_corrupted = True

            data.append({
                'filename': file,
                'path': file_path,
                'extension': os.path.splitext(file)[1].lower(),
                'size_bytes': file_size,
                'is_empty': is_empty,
                'is_corrupted': is_corrupted,
                'md5_hash': md5_hash
            })

    df = pd.DataFrame(data)

    # 3. Mark Duplicates
    # A file is a duplicate if its hash has appeared before in the dataset
    # (Excluding empty/corrupted files from hash-based duplicate logic if preferred)
    df['is_duplicate'] = df.duplicated(subset=['md5_hash'], keep='first') & df['md5_hash'].notnull()

    return df

# Usage
image_dir = 'corpus/cl_st1_ph1_tatiana_images'
df_images = get_image_details(image_dir)

# Quick summary of the findings
print(f"Total files processed: {len(df_images)}")
print(f"Empty files: {df_images['is_empty'].sum()}")
print(f"Corrupted files: {df_images['is_corrupted'].sum()}")
print(f"Duplicate files: {df_images['is_duplicate'].sum()}")

# Displaying the first few rows
df_images

Total files processed: 19758
Empty files: 285
Corrupted files: 1
Duplicate files: 3721


,filename,path,extension,size_bytes,is_empty,is_corrupted,md5_hash,is_duplicate
0,tweet_012498_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_012498_...,.jpg,138846,False,False,391d0d6daceb6d426d2078fb733390c8,False
1,tweet_010123_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_010123_...,.jpg,30159,False,False,b00292269d54e827ee9495808f3fc573,False
2,tweet_001669_000001.png,corpus/cl_st1_ph1_tatiana_images/tweet_001669_...,.png,283103,False,False,3ae0ee17300eb5ceb88cefb14c721056,False
3,tweet_010326_000001.png,corpus/cl_st1_ph1_tatiana_images/tweet_010326_...,.png,23994,False,False,894cf03d23637e5a66160fca5b73d996,False
4,tweet_026044_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_026044_...,.jpg,239387,False,False,3c59010e6a50b8c66a126bf30ff8ab9c,False
...,...,...,...,...,...,...,...,...
19753,tweet_020518_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_020518_...,.jpg,51436,False,False,4d13b9da8757b9ff400234697d93e36c,False
19754,tweet_019052_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_019052_...,.jpg,52686,False,False,f3b55cfeeeb911cafffcd29bcaad111d,False
19755,tweet_002291_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_002291_...,.jpg,124357,False,False,866be5664bc978e99bab39da0ac9791d,False
19756,tweet_000064_000001.jpg,corpus/cl_st1_ph1_tatiana_images/tweet_000064_...,.jpg,173422,False,False,61138483864ea5a817bc224395834e28,False


## Copy deduplicated images to a separate directory

In [4]:
import os
import shutil

# 1. Define paths
output_dir = 'corpus/deduplicated_1'

# 2. Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# 3. Filter the DataFrame for valid files only
# Criteria: Not empty, Not corrupted, and Not a duplicate
df_valid = df_images[
    (df_images['is_empty'] == False) &
    (df_images['is_corrupted'] == False) &
    (df_images['is_duplicate'] == False)
    ]

print(f"Copying {len(df_valid)} valid files to {output_dir}...")

# 4. Copy the files
for _, row in df_valid.iterrows():
    src = row['path']
    dst = os.path.join(output_dir, row['filename'])

    try:
        shutil.copy2(src, dst)
    except Exception as e:
        print(f"Error copying {row['filename']}: {e}")

print("Task complete! Your deduplicated corpus is ready.")

Copying 15751 valid files to corpus/deduplicated_1...
Task complete! Your deduplicated corpus is ready.


## Export to a file

In [5]:
df_images.to_json("corpus/df_images.jsonl", orient='records', lines=True)

In [6]:
df_images.to_excel("corpus/df_images.xlsx", index=False)

## Find all subdirectories in `corpus/near_duplicates` that contain more than one file

```
(my_env) eyamrog@eyamrog-iMac:~/PycharmProjects/cl_st1_tatiana$ find corpus/near_duplicates -mindepth 1 -type d | while read dir; \
 do [ $(ls -1A "$dir" | wc -l) -gt 1 ] && echo "$dir"; done
corpus/near_duplicates/1010110110101010101000101010101110100011111010001011001110101100
corpus/near_duplicates/1001010110011000100100111011100110001100100111001011110010110000
corpus/near_duplicates/1101001011010010110110111101100111010001011000010110010101100101
corpus/near_duplicates/1100110011000100110011001100110010010011001100111011001111001011
corpus/near_duplicates/0111011100110011000101110011010100001101110000111101000111011101
corpus/near_duplicates/1111010011110101011010010010110101101001010010010111100101111001
corpus/near_duplicates/0011011000110110011000100010011000100111111000001110010010100100
corpus/near_duplicates/0011111100110110000111111001101100000111010011111010001110110011
corpus/near_duplicates/1011001001100100111100011111000010100110001010110111010101111100
corpus/near_duplicates/1111000011110000110100001011000111110101110100011100100111101000
corpus/near_duplicates/0010111100101101000111000000111000100110001011100001101010011011
corpus/near_duplicates/1001100111001001010011010000110000111000001001110011110000111000
corpus/near_duplicates/0100110101100100110010100001100000010101001100110011000101100000
corpus/near_duplicates/0110101111110010111000101100010011001100111011001100111011101100
corpus/near_duplicates/1100110001000110010001100100011000001000100111001000011010001110
corpus/near_duplicates/1011100010101000011010001111001011011110110011101110001111100011
corpus/near_duplicates/1110100011110010111100111110111011011011100110111100101111000010
corpus/near_duplicates/0101110001011000100100101011001011011010110100101010000010010001
corpus/near_duplicates/0100101100101011001110110101101011011111111111110110111001100110
corpus/near_duplicates/1000000110101001001011010000110110101010110010110100000111000001
corpus/near_duplicates/1011110000101000011100001111111011001111111000111110101111101010
corpus/near_duplicates/1111000000111100001111001001011010110110001100110111001101100001
corpus/near_duplicates/1001100111110011110100111101011111100111111001111110011110111111
corpus/near_duplicates/1000111000001101011100111100100110010001100010100101110001011110
corpus/near_duplicates/1011100001101000111100101101111011001111111000111110101111001010
corpus/near_duplicates/1011000000001110001011101010011010000110101010101011000110000010
corpus/near_duplicates/1010110010101100101011001010110011001110100001110001010101000101
corpus/near_duplicates/1101100010111100101101001111110011011000101100100101100011011010
corpus/near_duplicates/1000011110010111100001111101001111010011110001111100001111100001
corpus/near_duplicates/1101100011101100111100101011000011110110111100101010100001010100
corpus/near_duplicates/1000111000000110110011101001011010010111000100110011001100110001
corpus/near_duplicates/1100011011001111110110111100011111001011000110110011000101010011
corpus/near_duplicates/0001100011000010101100100011100000111000001100100001101001001100
corpus/near_duplicates/1110110011000010110001111000011110000111101001100000111100011001
corpus/near_duplicates/1001011010011011010110000101110000110101101001011011010111110010
corpus/near_duplicates/1100001010000100101001010010011100100111001011011010011010100100
corpus/near_duplicates/0110000001100000111101001101010110110100101101000011010000110100
corpus/near_duplicates/1100000001101001011000010011000001011101110101011111010011000100
corpus/near_duplicates/0000000001011001111100011101100111011001100110011111100100011001
corpus/near_duplicates/1110110111100010111000001110110011101100110011001101110001111100
corpus/near_duplicates/1110110001001100010111000111100101101101000110011101100111110001
corpus/near_duplicates/1111110001101100011001001111001001100010111110001111000011110010
corpus/near_duplicates/0000001100100011000100110001001100010111001011010000110000011000
corpus/near_duplicates/0100110100110011100010110001101101010001110000110010001101011010
corpus/near_duplicates/1110001010110010001100010111000101000001010001110101101100011011
corpus/near_duplicates/1010001110100111011001100111000011011010111101000110100011010010
corpus/near_duplicates/1001000000001110101011101010011010001110101010011011000110001010
corpus/near_duplicates/0011011101011011001101100010010000100100101011011101100011111000
corpus/near_duplicates/0100111001001100010101000001010000010110000110100010101100100011
corpus/near_duplicates/0111100101111000111110011110010110000100101001100000110100011100
corpus/near_duplicates/0010110100100110110011101100111010011011000110110001001100110011
corpus/near_duplicates/0001000010010100100101001101010011010100100101101001010011010100
corpus/near_duplicates/1010010001110010110001011110011001001000000010000001110101010100
corpus/near_duplicates/1111000011110000111110001110100011101000111110001111000011110000
corpus/near_duplicates/1011001010110010101100001011101001101110111001100110001010110010
corpus/near_duplicates/0011000000110100011110101111001110110110101001101110010111101101
corpus/near_duplicates/0110011011001110110010111110000011101100110010010001100000110011
corpus/near_duplicates/1110011011100010111010101110011011001011100110011001100110011001
corpus/near_duplicates/1100110011001101110011011100110011000111000100111001000110010001
corpus/near_duplicates/0000011011000000110100001010001110100010101000001000100110010011
corpus/near_duplicates/1100110101100101100101101001000100011011100010111100101101001101
corpus/near_duplicates/1001100111011100101110001111001110011001111100011000110110001001
corpus/near_duplicates/1011001001110000011111000101100011011001110101110100100111000101
corpus/near_duplicates/1100110011010110110101001001010001110001101100101100001011010010
corpus/near_duplicates/1111001001110000011110000101110011011001110101110100100111001101
corpus/near_duplicates/1111000011110000111100001110010011111100111100011100001111001110
corpus/near_duplicates/1001110011010000010100001100000011000000110010001100000110100101
corpus/near_duplicates/1111101011110010101100001011010010110000001101101111011011011000
corpus/near_duplicates/0111000001001010010110100000111001001001110110011111100111100111
corpus/near_duplicates/0101000100110011001100110011001100110011010011010100110110101010
corpus/near_duplicates/1110100010010100100001101000101011001110111001101100111110011001
corpus/near_duplicates/0100110001001101010011010100010000010110100101100001011100110010
corpus/near_duplicates/0110100101111111111111111011110100110111110101100011111000110111
corpus/near_duplicates/1111011011100110110000111100011011100110111000001110110111001001
corpus/near_duplicates/1101010011001010110100001101100110010010000100100101001101001000
corpus/near_duplicates/0011100100110001011010010110100101101001011010010110100100001111
corpus/near_duplicates/1101111011001100110011001100100111010001111100001111010111010000
corpus/near_duplicates/0000111100100111011001011101001111100011111100111110001111100011
corpus/near_duplicates/1001100010011000101011000010110000101100001011000110011001110110
corpus/near_duplicates/1111100111110001111110111100011111100101011000011110001111000001
corpus/near_duplicates/1111110000110011110011001110100011100000111111001111000011110000
corpus/near_duplicates/0000100001011000100000101001001011010000011100100111001001110100
corpus/near_duplicates/1101000011010010000100001010000110101011010011100011100011100010
corpus/near_duplicates/1111010011110001001011010110110101101001010010010111100101111001
corpus/near_duplicates/1100000011010010111100101111100010001100110011000000001100000001
corpus/near_duplicates/0101001010010110010100100110001101111010010000101100011010000111
corpus/near_duplicates/1010101010100011101001011010010010001100101001011000000110000100
corpus/near_duplicates/0010110000001101100101110111011101110111101001100111001111110011
corpus/near_duplicates/1111000111011001011010000110110001101100001010000010110110000001
corpus/near_duplicates/0001111101011011110110011001110010011100100111001001101110011111
corpus/near_duplicates/0010011101010001010111011111011001010001011100001011100010110110
corpus/near_duplicates/1001110010110100111001001011010010110111001100110011000101110001
corpus/near_duplicates/1100100011011010001100000101010111011011101110001010100011101000
corpus/near_duplicates/1111101010011101101001011011110100011001110010011000110110100101
corpus/near_duplicates/0010110010110001111111011100111111000111111100011111000111100101
corpus/near_duplicates/0011101101110100010111000111010001110000010110000001100110011011
corpus/near_duplicates/1111001111110010000001010000110111011101100011000010011100100011
corpus/near_duplicates/1100001011000011110010111100110111000010110000101100000011001001
corpus/near_duplicates/1001011010100110101001101010011010101100001111000011101000101010
corpus/near_duplicates/1001001110010010011100111011001110011001110110011111001100010010
corpus/near_duplicates/1111010011111101110111101101111010011110100110100000110001001100
corpus/near_duplicates/1010110010101110111011100010011001101110011010010110100101000001
corpus/near_duplicates/1010010010011001011010111000101100110001010110111000000101110011
corpus/near_duplicates/1101001011010100110101001101000011010100100101001110010011110100
corpus/near_duplicates/1100100011011100110110001001011000110110001101100111000011110000
corpus/near_duplicates/1111100011111100110111101101111010011110100110100000111000001100
corpus/near_duplicates/0111011101100011010100110101001101000011000100110101001100010011
corpus/near_duplicates/1110010001100110010011101101101011000011110010101101000011110001
corpus/near_duplicates/1100000011010000110110001101011010011110111100101111001010000000
corpus/near_duplicates/1011000000110000001100000010010010010110110011000100110000001001
corpus/near_duplicates/1101100011001000110010001100100010101110110011100100001001001110
corpus/near_duplicates/1110110011110000111010001100010011000100111000001111000011011000
corpus/near_duplicates/1100010010110101110000111101100111001000010011000101110001011100
corpus/near_duplicates/1001011010011011010110110101100000110101101001011011010111110010
corpus/near_duplicates/1111110001110000111100001111110011111110110110000111100011111000
corpus/near_duplicates/1100110011000100111010011011100010110000101001011011111100110011
corpus/near_duplicates/1111100001011110011111111011101111101010101001101000111011001110
corpus/near_duplicates/1110000111111000101110000011100011111000101110101111000011111100
corpus/near_duplicates/0111100011001000100011001000010011000100111000001100000111010010
corpus/near_duplicates/1110000111001001010010010010110110110100101101000010010011100110
corpus/near_duplicates/1100000011000000110001001001000010010000101101001011010010011100
corpus/near_duplicates/1101001111111010111111001111010011100100110010001111100011111000
corpus/near_duplicates/1111001011110011111101011111000111110001110001001100110011001100
corpus/near_duplicates/0110100011110010110011101100111111100011111000111110101011101011
corpus/near_duplicates/0000011100110111111011001110010011101100101011101100110011001100
corpus/near_duplicates/1100000011010010111100101111100010001100110001000000001100000001
corpus/near_duplicates/1100101111001011111000001010101010000010111000001010010011000010
corpus/near_duplicates/0001101110011011010110010001101110101110100001101110001000100011
corpus/near_duplicates/1010101100110011101100100010110000110010110010100101100011001100
corpus/near_duplicates/1110011001101101110101101101000110011100111110001100100010000110
corpus/near_duplicates/1011100010011110111111110011100111101010111001101100111011001110
corpus/near_duplicates/1110101011110011111100011111001111100111110001001100010011000100
corpus/near_duplicates/0110010001101100011001100110100101100001011001010110100101101100
corpus/near_duplicates/0110001001100111010000101000011011100110011100101110011011100110
corpus/near_duplicates/1101111011010010110111101101101011101100110011011000011110000011
corpus/near_duplicates/0101100100111001011001111001110011110000111100011011001101011011
corpus/near_duplicates/0001101000010100011001010110010111000110000100110101001111000111
corpus/near_duplicates/0011100000110100011101000111010010010100100100001101001010010000
corpus/near_duplicates/1111010111110001001011010110110101101001010010010111100101111001
corpus/near_duplicates/0110001111100111110001011101100110010100110111000001110000011100
corpus/near_duplicates/1100110011011100110101001101110001001110000100110011001100110011
corpus/near_duplicates/0000000100000011000000100000001100010010000100100011100000111000
corpus/near_duplicates/1100011011001111110110111100011111001111100100110011100101110011
corpus/near_duplicates/1111110111001011100110111001101011011111001101111000011101000111
corpus/near_duplicates/1110100011101010111000101110001111100010111000111000110110001101
corpus/near_duplicates/1001111001001011001110111011011111100110110001101001111000111011
corpus/near_duplicates/1111110001011110101100111010101011100110100011101100110011110000
corpus/near_duplicates/0011001011110001110111001101100011011100110110001111100100110011
corpus/near_duplicates/1111000011110000111110001111100011101000111110001111000011110000
corpus/near_duplicates/0110000001100000111100001011100011000100100101001100110011010001
corpus/near_duplicates/1101100000011000001001000011110000111100001111000110011001100011
corpus/near_duplicates/1100110011100100111110011011100010110000101001011011111100110011
corpus/near_duplicates/0100011111010110110100101001101110011011110010101111100011100100
corpus/near_duplicates/1100000010000100101001010010011100101101101011101010010010100011
corpus/near_duplicates/1011101011010010110100001111000011110000111100001101000001101000
corpus/near_duplicates/0010110100001101100101110111011101110111101001100111001111110011
corpus/near_duplicates/1001100010011000101000001111100110110001111000001100011011101100
corpus/near_duplicates/1111010011111101110111101101111010011110000110100000110001001100
corpus/near_duplicates/1110000011100000011101000111100001111010011111100011110001110000
corpus/near_duplicates/1111100011111000111011001110110011101100111011001111100011111000
corpus/near_duplicates/1100110011100100111110011011100010010000101001011011111100110011
corpus/near_duplicates/0111001100110001011010010100100001001100010111100011101000100011
corpus/near_duplicates/1001111111011010100011111000110100001111010011000110110010110100
corpus/near_duplicates/1001110011100100011001100111000110010011101001011010011011001010
corpus/near_duplicates/1101110011100110001101100001110001000111101100001010010111010111
corpus/near_duplicates/0010010010100100101001001000100100101000001001010110010001110000
corpus/near_duplicates/1101100010111000001100001111001001010010010101100010110001001100
corpus/near_duplicates/0001001101100011011001110010111110101101101011011010000011000100
corpus/near_duplicates/1111000011111000111010001110100011101100111010001111100011110000
corpus/near_duplicates/1111001001110000011111000101100011010011110100110100010111000001
corpus/near_duplicates/1011100111110100111111000011111000111100001110000111100011110001
corpus/near_duplicates/0101011111100111001110110010111111010111011100110111001100011111
corpus/near_duplicates/1101110001011110100110101101110000111100001100000001110010100100
corpus/near_duplicates/1100011111001111110101111111010111010101110011011000110010111100
corpus/near_duplicates/1001010110110000001100010011000101110001011010010110100101110001
corpus/near_duplicates/1000110010010100100101001001010010010110001100100011010000110100
corpus/near_duplicates/1001011000101100001111000011110011001000100100101001101010011010
corpus/near_duplicates/0101111001010010001101001001011010010000110101001100011011010110
corpus/near_duplicates/0101001111100000111001001000110010110100111010101111000000011001
corpus/near_duplicates/0111100111100001110000010101110000001100110111000111100111001100
corpus/near_duplicates/0011111001011011110010111100001111000100110001001100110001001000
corpus/near_duplicates/0011001001011100111010010101111010111011011001101010010010011100
corpus/near_duplicates/1000000000100000110110101101011010010110110100000010100110010000
corpus/near_duplicates/1011100001011110011111111011101111101010101001101000111011001110
corpus/near_duplicates/0101111011001111011000101110101111110011101001111011111111110011
corpus/near_duplicates/1101000011000110110110101111101010110000111011001100100011111000
corpus/near_duplicates/1110110011111000101101001010010010010100111001010110001000100100
corpus/near_duplicates/0111111011011001010110010101110111011101100111011001110110011100
corpus/near_duplicates/1110010001111001001100011000100001011101110011011001100111000101
corpus/near_duplicates/1000001110100011001000111110001111000011110000111101110011111010
corpus/near_duplicates/1111000011011000110110001100100011000110110001111101001111000011
corpus/near_duplicates/1101100011110000111100001111000011100100111100011100000111001110
corpus/near_duplicates/1100000011010010111100101111100010011100110011001000011000000011
(my_env) eyamrog@eyamrog-iMac:~/PycharmProjects/cl_st1_tatiana$
```